In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": [],
      "gpuType": "T4"
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "accelerator": "GPU"
  },
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 02 — Scientific Evaluation & Live Side-by-Side Demo (Colab T4)\n",
        "**Owners:** Member 3 (Evaluation) & Member 4 (Demo UI)  \n",
        "**Prerequisite:** Trained LoRA adapter saved in `/content/drive/MyDrive/domain-expert-qlora/final_adapter`"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 1: Mount Drive & Sync Latest Code from GitHub\n",
        "import os\n",
        "from google.colab import drive\n",
        "drive.mount('/content/drive')\n",
        "\n",
        "GITHUB_USER = \"https://github.com/Saurav00752\"  # <-- Replace with your GitHub username\n",
        "REPO_NAME = \"domain-expert-qlora\"\n",
        "BRANCH = \"dev\"\n",
        "\n",
        "%cd /content\n",
        "if not os.path.exists(REPO_NAME):\n",
        "    !git clone -b {BRANCH} https://github.com/{GITHUB_USER}/{REPO_NAME}.git\n",
        "%cd /content/{REPO_NAME}\n",
        "!git checkout {BRANCH}\n",
        "!git pull origin {BRANCH}\n",
        "!pip install -q -r requirements.txt"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 2: Step 1 — Run Baseline ('Before') Evaluation on Held-Out test.jsonl\n",
        "!python evaluation/eval_baseline.py \\\n",
        "  --config training/configs/config.yaml \\\n",
        "  --test-file data/processed/test.jsonl \\\n",
        "  --output evaluation/results/baseline_metrics.json"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 3: Step 2 — Run Fine-Tuned ('After') Evaluation on Held-Out test.jsonl\n",
        "ADAPTER_DIR = \"/content/drive/MyDrive/domain-expert-qlora/final_adapter\"\n",
        "\n",
        "!python evaluation/eval_finetuned.py \\\n",
        "  --config training/configs/config.yaml \\\n",
        "  --adapter-dir {ADAPTER_DIR} \\\n",
        "  --test-file data/processed/test.jsonl \\\n",
        "  --output evaluation/results/finetuned_metrics.json"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 4: Step 3 — Generate Before vs. After Chart & Markdown Report\n",
        "import shutil\n",
        "from IPython.display import Image, Markdown, display\n",
        "\n",
        "!python evaluation/metrics_report.py\n",
        "\n",
        "display(Image(filename=\"evaluation/results/before_vs_after_chart.png\"))\n",
        "with open(\"evaluation/results/qualitative_samples.md\", \"r\", encoding=\"utf-8\") as f:\n",
        "    display(Markdown(f.read()))\n",
        "\n",
        "# Backup evaluation results to Google Drive so Member 3 can commit them to GitHub\n",
        "drive_eval_dir = \"/content/drive/MyDrive/domain-expert-qlora/eval_results\"\n",
        "os.makedirs(drive_eval_dir, exist_ok=True)\n",
        "for fname in [\"baseline_metrics.json\", \"finetuned_metrics.json\", \"before_vs_after_chart.png\", \"qualitative_samples.md\"]:\n",
        "    shutil.copy(f\"evaluation/results/{fname}\", f\"{drive_eval_dir}/{fname}\")\n",
        "print(f\"[BACKUP COMPLETE] Evaluation artifacts copied to {drive_eval_dir}\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 5: Step 4 — Launch Live Side-by-Side Gradio Demo (Generates Public Share URL)\n",
        "ADAPTER_DIR = \"/content/drive/MyDrive/domain-expert-qlora/final_adapter\"\n",
        "!python app/ui.py --config training/configs/config.yaml --adapter-dir {ADAPTER_DIR} --share"
      ]
    }
  ]
}